<a href="https://colab.research.google.com/github/Nawaf-Rayhan585/YOLO_Projects/blob/main/rat-rodent-detection/train_rat_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rat/Rodent Detection Model — Train on Colab T4, Use Anywhere

This notebook trains a YOLOv8 model to detect and classify rats/rodents in images and video.

**How to use:**
1. Runtime -> Change runtime type -> GPU -> T4
2. Run cells top to bottom
3. At the end, download `best.pt` and drop it into `rat_detect.py` in this folder

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install packages

In [ ]:
!pip install ultralytics roboflow -q

## 3. Get a rat/rodent dataset

We use Roboflow Universe, it has ready-made rat detection datasets.

1. Go to https://universe.roboflow.com and search "rat detection" or "rodent detection"
2. Pick a dataset, click "Download this Dataset" -> format **YOLOv8** -> copy the code snippet
3. It gives you an API key + workspace + project + version, paste them below

If you already have your own dataset in YOLO format, skip this and just upload it to `/content/dataset` with a `data.yaml` inside.

In [ ]:
from roboflow import Roboflow

# paste your own values here, get them from roboflow universe dataset page
ROBOFLOW_API_KEY = "YOUR_API_KEY"
WORKSPACE = "YOUR_WORKSPACE"
PROJECT = "YOUR_PROJECT"
VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(dataset.location)

## 4. Train the model (uses Colab's T4 GPU)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano, fast and light, good for T4 and later for your PC

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    project="rat_detector",
    name="train1",
)

## 5. Validate the model

In [ ]:
metrics = model.val()
print(metrics)

## 6. Test on 2-3 images

Upload a few rat images to `/content/test_images/` (use the folder icon on the left, or the upload cell below), then run.

In [ ]:
from google.colab import files
import os

os.makedirs("/content/test_images", exist_ok=True)
uploaded = files.upload()  # pick 2-3 rat images from your PC

for name in uploaded.keys():
    os.rename(name, f"/content/test_images/{name}")

In [ ]:
best_model = YOLO("rat_detector/train1/weights/best.pt")

test_results = best_model.predict(
    source="/content/test_images",
    conf=0.4,
    save=True,
)

print("check results in runs/detect/predict or rat_detector/train1/predict")

## 7. Download your trained model

In [ ]:
from google.colab import files

files.download("rat_detector/train1/weights/best.pt")

## 8. Use `best.pt` locally

Drop the downloaded `best.pt` into this folder and run:

```bash
python rat_detect.py --model best.pt --source your_video.mp4
```

Your PC does NOT need a GPU to just run this model, CPU is fine for normal use.